# 03 · 从零建立 SNN → 拟合 → 传播与 Z/M（Fig4/5、Supplementary Video 2）

这里 **新采样神经元与连接、新生成 OU/Poisson 输入、积分膜电位/突触、新检测传播事件**。不载入网络缓存、仿真结果、现成评分或 GIF。模型拟合输入是临床训练集的逐事件质心与固定触点坐标；目标函数在本 notebook 中重新拟合。读取患者观测用于拟合，与读取仿真结果画图是不同步骤。

模型：20×20 mm、100 cells/mm²=40,000 cells（80% E/20% I）、dt=0.1 ms、延迟 E/I LIF。core 只降低 E 阈值；core 内独立 OU + Poisson，外部 E 与 I 接受确定性期望输入，无外部事件 kick。接触点位置是患者几何先验，不能据此认定 SOZ 回路。

默认保留神经元数，只用 12 秒/候选、K=1..4 各 2 次评估、1 次 restart，并单独扫描角度。完整尺度参数在 `config/snn_paper.json`（60 秒、31 次评估、4 restarts、13 角度）。**notebook 的 Sobol+局部精修优化器不同于原历史 delayed GP/qLogEI；复制的原优化调度器保留在 modeling/scripts/topic4_core_count_optimization。** 新运行结果不能宣称等于原 4×31 曲线、83/413 密度或25人结果。

预计需要数十分钟 CPU；最多4个本地进程，运行状态持续写入本次输出目录。已验证的 Numba 加速只替换有序 scatter 加法，未改变随机数和方程；40,000-cell 6秒轨迹与原CPU实现逐数组一致。

In [ ]:
from pathlib import Path
import sys, os, json
# Only insert this independently copied package, never the original repository.
CODE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(CODE) not in sys.path: sys.path.insert(0, str(CODE))
from tools.runtime_guard import install_guard, local_import_report
install_guard()
import numpy as np
import matplotlib.pyplot as plt
from paper_paths import CODE_ROOT, INPUT_ROOT, OUTPUT_ROOT, sha256
from workflows.common import new_session, publish_session, previous_session, write_json
from plotting import computational as plot
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
from modeling.computational import (load_configuration,build_substrate,simulate,fit_objective,
    score_simulation,save_run)
from modeling.experiments import run_experiments
from analysis.from_events import cluster_events
from analysis.spectral_fields import energy_field_trajectory,native_entry
from analysis.src.interictal_propagation import assign_events_to_templates
from scipy.stats import rankdata
config = load_configuration(CODE_ROOT/'config'/os.environ.get('HFO_SNN_CONFIG','snn_tutorial.json'))
OUT=new_session('snn'); FIG=OUT/'figures'; print('输出:',OUT,flush=True)
with np.load(INPUT_ROOT/'computational/E1146_training_observations.npz',allow_pickle=False) as z:
    patient = {k:z[k] for k in z.files}
names=patient['contact_names'].tolist(); contacts=patient['contact_xy_mm']
observer=json.loads((INPUT_ROOT/'computational/E1146_observer.json').read_text())
assert observer['contact_names']==names
print(config['scope']);print(config['parameters']);print('patient FIT/CAL:',patient['fit_centroid_ms'].shape,patient['cal_centroid_ms'].shape)

get_ipython().run_line_magic('matplotlib', 'inline')


**Fig4A：模型设定与初始化。** 新采样位置与无自连接的空间 E/I 图；E→E 椭圆核与传导延迟由物理坐标计算。每个神经元固定阈值分位数，候选仅改变 core 成员；不能为每个候选重新抽有利阈值。这个独立入口重新采样的图身份不等于历史缓存图，所以精确历史事件时间不应被预设。

In [ ]:
substrate=build_substrate(config)
print('新建神经元数:',substrate['net']['NE']+substrate['net']['NI'],flush=True)
# No precomputed graph loaded. Inspect this configuration before increasing the budget.
write_json(OUT/'model_configuration.json',config)


**Fig4B：拟合目标。** 从独立 patient FIT 建立三个分布特征：rank pattern、局部/跨杆时间、participation。按原来记录内连续16事件的256组 CAL 抽样索引重新校准量纲（输入仅保存样本索引，不保存得分）；三个分量等权，无类别标签输入。`J` 是保留有限样本修正的有符号统计量，不强行截成正值。少于16个合格事件或 runaway 记为不可评分。

In [ ]:
objective=fit_objective(patient['fit_centroid_ms'],patient['cal_centroid_ms'],names,
                        n_fourier=config['n_fourier'],cal_draw_indices=patient['cal_draw_indices'])
print('新拟合目标尺度:',objective.scales,flush=True)
write_json(OUT/'objective_configuration.json',dict(version=objective.version,scales=objective.scales,n_fourier=config['n_fourier']))


**Fig4F/G 的原料：一次新的基础仿真。** 波形是 Gaussian E-spike readout，不是假称电压；动态场记录每 2 ms、1 mm² 中活动 E 细胞数。观察器由独立参考固定，不能跟随候选调到“有事件”。

In [ ]:
baseline=simulate(substrate,config['centers_mm'],contacts,seed=config['training_seed'])
score_simulation(baseline,observer,objective,config['burnin_ms'])
save_run(OUT/'baseline_new_simulation.npz',baseline)
write_json(OUT/'baseline_score.json',baseline['score'])
print('本次仿真:',baseline['physical_status'],baseline['wall_seconds'],'s',baseline['score'],flush=True)
plot.save(plot.substrate_panel(baseline,config['radius_mm']),FIG,'fig4_A_model','新建立的core阈值场、触点读出位置与新仿真的E/I群体发放率。')
write_json(OUT/'baseline_metadata.json',{k:v for k,v in baseline.items() if not isinstance(v,np.ndarray) and k!='slow'})
plot.save(plot.bandpassed_model_waveform(baseline),FIG,'fig4_F_baseline_waveforms','从新仿真spike计算的触点包络。读出单位与真实SEEG电压不同。')
plt.show()


**Fig4C/D/E、Fig5A：执行有限实验预算。** 每个候选都积分新轨迹。K=1..4 使用相同评估次数；角度改变连接采样方向，不能只旋转图片。独立 noise 的确认不反馈训练。Z/M 与基础网络共用位置、连接、阈值和随机种子，只增加慢变量；采用当前 Fig5 历史算术和参数，τz=5 s、τm=0.5 s。慢变量的高发放率保护停止与“发作振荡”是不同概念。

In [ ]:
experiments=run_experiments(substrate,contacts,observer,objective,OUT/'experiments')
searches=[value for name,value in experiments.items() if name.startswith('core')]
rows=[r for search in searches for r in search['rows']]
print('新完成的候选:',len(rows),'可评分:',sum(r['J'] is not None for r in rows))


**Fig4B：新候选的三项可观察误差。** 分别计算平均归一化rank绝对误差、杆内先后概率误差及参与率误差，并按杆等权汇总。它们不同于校准后用于优化的J分量；不可用旧评分补点。

In [ ]:
plot.save(plot.raw_errors(rows),FIG,'fig4_B_observable_errors','本次候选的平均rank、杆内先后概率及participation误差。与用于优化的J分量分别保存。');plt.show()


**Fig4C：core 数与优化曲线。** best-so-far 单调下降是优化记录的定义，不是独立机制证据。默认每K两个评估只是可运行示范；四次 restart 的均值/样本SD需要完整预算。

In [ ]:
plot.save(plot.optimization_curves(searches),FIG,'fig4_C_core_search','K=1至4的相同预算新优化。横轴为实际新评估次数，未达到事件数的候选不造损失。');plt.show()


**Fig4D：方向响应。** 固定其他参数和noise，只改变全局 E→E 核角度重新构图/仿真。默认3点；论文规模配置13点。

In [ ]:
angle_rows=[dict(angle=0.,**baseline['score'])]+[r for name,value in experiments.items() if name.startswith('angle_') for r in value]
angle_rows=sorted(angle_rows,key=lambda r:r['angle'])
plot.save(plot.raw_errors(angle_rows,angle=True),FIG,'fig4_D_direction_response','固定其他物理量和噪声，重新仿真角度响应并计算三项可观察误差。缺失量保持不可估。');plt.show()


**Fig4E：低误差 core 位置。** 只用本次有限J配置的最低20%，每个配置保留全部core位置。小预算密度只能展示计算方法，不能替代论文的83/413统计。

In [ ]:
plot.save(plot.core_density(rows,config['radius_mm']),FIG,'fig4_E_core_positions','本次新搜索最低训练误差20%的core位置分布。当前候选数量与正式论文不同。');plt.show()


**Fig4G：患者与模型的平均传播rank。** 从患者FIT重新拟合模板，再给本次模型事件分配模式并计算平均归一化rank。缺席触点为NaN，不能按−1或有限phantom rank参与平均。这个图属于拟合后的描述性比较。

In [ ]:
from modeling.panel_analysis import model_patient_comparison
contact_contract=json.loads((INPUT_ROOT/'computational/E1146_contact_contract.json').read_text())
matching=model_patient_comparison(baseline['primary_centroids_ms'],patient['fit_centroid_ms'],names,contact_contract,n_null=1000)
plot.save(plot.model_rank_profiles(matching,names),FIG,'fig4_G_mean_rank','模型新事件与患者FIT的平均传播rank。缺席触点保留缺失，模板从输入观测重新拟合。');plt.show()


**Fig4H：触点交叉匹配矩阵。** 按固定几何合同在每根杆内交替拆分触点：一半用于给模型事件分配患者模板，另一半计算相关，再交换两半。within-shaft contact shuffle 对每个事件重复相同置换，并重做整个分配/评分过程，避免用评分触点同时选择匹配类别。

In [ ]:
plot.save(plot.model_crossfit_matrix(matching),FIG,'fig4_H_crossfit','在未参与模板分配的另一半触点上评价Spearman相关。当前只有一个新网络，不能称为25名患者队列结果。');plt.show()
print('两条语义匹配的within-shaft null p:',matching['diagonal_p'])
write_json(OUT/'model_patient_matching.json',{k:v for k,v in matching.items() if k!='within_shaft_null'})
np.savez_compressed(OUT/'model_patient_matching_null.npz',null=matching['within_shaft_null'])


**Supplementary Video 2：从新 spike 生成传播图和 GIF。** 对两个新模拟工作点按本次 J 排序；患者 FIT 重新拟合K=2模板，只用于描述新事件方向，不参与优化损失。每个工作点取各模板类别按时间出现的第一个合格事件。两个事件独立对齐自身最早质心，绝不声称同时发生。如果任何工作点没有足够可评分事件或未出现两种类别，明确报告不可生成该四事件比较，不拿缓存结果补图。

In [ ]:
def centroid_events(table):
    mask=np.isfinite(table); rank=np.full(table.shape,-1,int)
    for i,row in enumerate(table): rank[i,mask[i]]=rankdata(row[mask[i]],method='dense').astype(int)-1
    return dict(ranks=rank.T,participating=mask.T,channel_names=np.asarray(names))
patient_templates=cluster_events(centroid_events(patient['fit_centroid_ms']))
comparison=[baseline,experiments['outward_100']]
comparison=sorted([run for run in comparison if run['score']['J'] is not None],key=lambda run:-run['score']['J'])
selected=[]
for run in comparison:
    table=np.asarray(run['observed']['centroid_ms']); e=centroid_events(table)
    labels=assign_events_to_templates(e['ranks'],e['participating'],patient_templates['templates'])
    eligible=run['observed']['primary_event_indices']
    found=[[i for i in eligible if labels[i]==k and run['observed']['events'][i]['window_ms'][0]>=config['burnin_ms']] for k in range(2)]
    selected.append([v[0] for v in found] if all(found) else None)
movie_status='NOT_ESTIMABLE'
if len(comparison)==2 and all(v is not None for v in selected):
    plot.propagation_movie(comparison,FIG/'supplementary_video2_new.gif',config['radius_mm'],event_indices=selected)
    movie_status='GENERATED_FROM_NEW_SPIKES'
    from IPython.display import Image,display
    display(Image(filename=str(FIG/'supplementary_video2_new.png')))
else:
    print('本次比较缺少两个可评分工作点或完整两类事件；不使用旧 GIF。')
print('Video 2:',movie_status)


**Fig4I：独立确认与队列边界。** 本次独立noise检验同一个患者先验模型，不是新患者。Fig4I的患者队列需分别提供每人的观测表/几何并运行优化，不能把不同种子当患者。包内 `modeling/cohort.py` 提供相同的逐患者执行入口。

In [ ]:
confirmation=experiments['new_noise_confirmation']
print('独立 noise（不反馈优化）:',confirmation['score'])
write_json(OUT/'independent_noise_confirmation.json',confirmation['score'])


**Fig5A：同一个新网络启用Z/M，读取波形。** 按当前六面板版本分开A波形/B慢变量/C空间图/D三维轨迹/E起始场。这里显示原始突触电流代理，保留DC，按固定1.5–3s基线P95定标；采样2000Hz。该基本例的未滤波显示不同于候选图10kHz采样后的30–1000Hz显示，不能混写带宽。历史t120与严格200Hz/200ms进入点分别计算；保护触发后额外记录1.5s以保留完整进入后功率窗。

In [ ]:
from analysis.model_transition import display_entry,onset_broadband_field,compare_raw_patient_fields
zm=experiments['zm_enabled']
entry=native_entry(zm['time_ms'],zm['rate_E'],dt_ms=config['parameters']['dt'])
onset=display_entry(zm)
print('t120:',onset,'native entry:',entry,'status:',zm['physical_status'])
plot.save(plot.raw_current_waveforms(zm,onset=onset),FIG,'fig5_A_waveforms','新Z/M轨迹的未滤波突触电流代理，保留持续电平。此示例采样2000Hz，按固定基线P95定标。');plt.show()


**Fig5B：慢变量。** 从同一次积分取全体E细胞的平均Z和有效适应电流ηM；这里不把M误作阈值，也不将有效慢变量当成实测离子浓度。本基本例只记录All E均值，不伪造Core A曲线。

In [ ]:
plot.save(plot.slow_variables(zm,config['zm']['eta_m']),FIG,'fig5_B_slow_variables','同一新轨迹的全体E平均抑制效能Z及有效适应电流ηM。');plt.show()


**Fig5C：空间传播快照。** 对新生成的2ms E活动网格作50ms平均。若出现t120，固定取前后−0.3、−0.1、+0.1、+0.3秒；否则固定等间隔展示，不挑选看起来最好的传播事件。

In [ ]:
plot.save(plot.activity_snapshots(zm,config['radius_mm'],onset=onset),FIG,'fig5_C_spatial_activity','同一新轨迹四个预定时刻的50ms平均E活动空间图。');plt.show()


**Fig5D：三维状态轨迹。** 原始全体E发放量先按5ms分箱并作σ=1bin平滑，与同一时刻Z、ηM配对。单条状态轨迹不等于分岔图或吸引子证据。

In [ ]:
plot.save(plot.state_trajectory(zm,config['zm']['eta_m']),FIG,'fig5_D_state_trajectory','同一新仿真的Z、有效适应电流ηM及全体E发放率三维轨迹。');plt.show()


**Fig5E：进入后宽频能量场。** 从新电流在全部15触点上作CAR；1s Tukey0.25窗、0.5s步长、1–150Hz对数PSD和0.5–3.5s基线robust-z，进入后1s必须完整。模型基线允许5个完整重叠谱窗，与临床远端基线的≥50窗区别明确。场的轴/核宽/支持从固定interictal rank与坐标计算；本图显示带符号z，不把负增强改画为正。

In [ ]:
energy=energy_field_trajectory(zm['contact_current'],2000.,contacts,
    patient_templates['templates'][0],patient_templates['support'][0],baseline_sec=(1.5,3.),band=(30,80))
onset_field=onset_broadband_field(zm['contact_current'],2000.,onset)
plot.save(plot.onset_energy_field(zm,onset_field,energy),FIG,'fig5_E_model_onset_field','从新电流重新计算进入后1–150Hz能量，并在固定interictal平面绘制带符号robust-z场。');plt.show()
write_json(OUT/'new_onset_broadband_field.json',onset_field)
write_json(OUT/'fixed_energy_geometry.json',dict(plane=energy['plane'],contact_mask=energy['contact_mask']))
print('模型起始场:',onset_field['status'])


**Fig5E，患者比较接口。** 若提供clinical_examples.json，每条记录指向原始SEEG NPZ及onset/真实远端baseline，逐次重算10s临床场，再应用正平均增强门槛、最大signed identity-field相关和编号打破平局。输入需完整CAR触点，不读现成患者能量场。这是选择后的描述例，不给独立验证p值；默认示例未附这组原始发作数据。

In [ ]:
clinical_status='NOT_RUN_REQUIRES_RAW_ICTAL_EXAMPLES'
clinical_manifest=INPUT_ROOT/'computational/clinical_examples.json'
if clinical_manifest.exists():
    raw_seizures=[]
    for item in json.loads(clinical_manifest.read_text()):
        with np.load(clinical_manifest.parent/item['file'],allow_pickle=False) as z:
            raw=dict(samples_V=z['samples_V'],sfreq=float(z['sfreq']),channel_names=z['channel_names'].tolist())
        raw.update({k:item[k] for k in ['onset_sec','baseline_sec','seizure_index']});raw_seizures.append(raw)
    clinical_comparison=compare_raw_patient_fields(onset_field,raw_seizures,names,energy['scorer'])
    write_json(OUT/'new_patient_onset_comparison.json',clinical_comparison);clinical_status=clinical_comparison['status']
    if clinical_comparison['selected'] is not None:
        selected_patient_field=dict(status='OK',robust_z=clinical_comparison['selected']['robust_z'])
        plot.save(plot.onset_energy_field(zm,selected_patient_field,energy,label='selected patient'),FIG,'fig5_E_selected_patient_field','从原始发作信号重算并按明确规则选出的描述性例图。选择不等于独立验证。');plt.show()
print('临床例图:',clinical_status)


**补充计算：gamma能量和场评分随时间变化。** 另用30–80Hz、100ms Hann窗、10ms步长、1.5–3s固定基线得到robust-z，再用固定interictal场计算signed r/q。横向镜像围绕中心化后的模板轴；相关和投影采用同一个|r|选择。它们与上面的宽频起始场分别保存。

In [ ]:
plot.save(plot.energy_alignment(energy),FIG,'supplementary_gamma_field_scores','从新电流计算的gamma能量、固定场相关及投影时间序列。');plt.show()
np.savez_compressed(OUT/'new_energy_fields.npz',time_sec=energy['time_sec'],power=energy['power'],robust_z=energy['robust_z'],**energy['scores'])


**Fig5F：临床队列边界与报告。** 当前13人/140次dB统计需要完整临床原始输入、phase/频带、发作资格与患者内汇总。不能用本次SNN、单例或robust-z去替代该队列dB结果。默认例明确记录未执行完整队列；相关基础计算在analysis/spectral_fields.py，方法接口文档保留定义差异。

In [ ]:
input_identity={name:sha256(INPUT_ROOT/'computational'/name) for name in ['E1146_training_observations.npz','E1146_observer.json','E1146_contact_contract.json']}
report=dict(input_sha256=input_identity,scope=config['scope'],configuration=config,baseline_score=baseline['score'],
            core_evaluations=len(rows),core_scorable=sum(r['J'] is not None for r in rows),video2=movie_status,
            entry=entry,historical_t120_sec=onset,model_onset_field=onset_field['status'],clinical_fig5E=clinical_status,
            clinical_fig5F='NOT_RUN_REQUIRES_RAW_COHORT',fig4I='ONE_PATIENT_NEW_NOISE_ONLY',imports=local_import_report())
write_json(FIG/'video_selection.json',dict(rule='first eligible chronological event per freshly fitted patient template',events=selected,status=movie_status))
publish_session('snn',OUT,report);print('全部本次输出:',OUT)
